# BERDL Lakehouse — Ingest: {DATASET}

Uploads source TSV files to MinIO bronze, builds a config JSON, and runs `ingest()` to write Delta tables to the personal silver warehouse.

**Workflow:**
1. Fill in `DATA_DIR`, `USER`, `DATASET`, `BRONZE_DEST`, and `TABLES` in the **Configuration** cell.
2. Run all cells through **Pre-flight** and review the printed plan.
3. Set `CONFIRMED = True` in Configuration, then re-run from the Pre-flight cell onward.

### Configuration

In [ ]:
from pathlib import Path

# ── USER CONFIGURATION ────────────────────────────────────────────────────────
DATA_DIR    = Path("{/absolute/path/to/your/source/data}")  # local directory containing source files
USER        = "{your_minio_username}"                        # MinIO username (e.g. amkhan)
DATASET     = "{your_dataset_name}"                          # → namespace: u_<USER>__<DATASET>
BUCKET      = "cdm-lake"
BRONZE_DEST = f"users-general-warehouse/{USER}/datasets/{DATASET}"  # MinIO path (no s3a://)
MODE        = "overwrite"                                    # "overwrite" or "append"
CONFIRMED   = False
# ─────────────────────────────────────────────────────────────────────────────

# Table definitions: one entry per source file.
# name       — Delta table name that will be created in the namespace
# source     — filename inside DATA_DIR (TSV expected; adjust defaults below for CSV)
# schema_sql — Spark SQL column definitions; supported types: STRING INT BIGINT DOUBLE FLOAT BOOLEAN
#              Set to "" to infer all columns as STRING
TABLES = [
    {
        "name":       "{table_name}",
        "source":     "{source_file.tsv}",
        "schema_sql": "{col1} STRING, {col2} STRING",
    },
    # Add more tables as needed
]

# ── Derived paths (no edits needed below this line) ──────────────────────────
NAMESPACE   = f"u_{USER}__{DATASET}"
BRONZE_BASE = f"s3a://{BUCKET}/{BRONZE_DEST}"
SILVER_BASE = f"s3a://{BUCKET}/users-sql-warehouse/{USER}/{NAMESPACE}.db"
CONFIG_DIR  = Path.home() / ".data_lakehouse" / "configs"
CONFIG_PATH = CONFIG_DIR / f"{DATASET}_ingest.json"

print(f"Namespace : {NAMESPACE}")
print(f"Bronze    : {BRONZE_BASE}")
print(f"Silver    : {SILVER_BASE}")
print(f"Tables    : {[t['name'] for t in TABLES]}")
print(f"Mode      : {MODE}")

### Imports and `berdl_notebook_utils` stubs

Replaces JupyterHub-only submodules with lightweight stubs so `data_lakehouse_ingest` can be imported from a local machine.

**Why this is needed:** `data_lakehouse_ingest` imports from `berdl_notebook_utils`, a package that only exists on the BERDL JupyterHub cluster. When running locally, all submodules must be stubbed via `sys.modules` *before* the import, otherwise the import fails with `ModuleNotFoundError`.

In [ ]:
import io, json, logging, os, subprocess, sys, time
from types import ModuleType

# Why these stubs are needed:
# data_lakehouse_ingest imports berdl_notebook_utils at the module level.
# That package only exists on the BERDL JupyterHub cluster, so running locally
# raises ModuleNotFoundError unless all submodules are stubbed first.
_STUB_MODULES = [
    "berdl_notebook_utils",
    "berdl_notebook_utils.berdl_settings",
    "berdl_notebook_utils.clients",
    "berdl_notebook_utils.setup_spark_session",
    "berdl_notebook_utils.spark",
    "berdl_notebook_utils.spark.database",
    "berdl_notebook_utils.spark.cluster",
    "berdl_notebook_utils.spark.dataframe",
    "berdl_notebook_utils.minio_governance",
]
for _name in _STUB_MODULES:
    sys.modules[_name] = ModuleType(_name)

# The MinIO client stub is wired after the Minio object is created below.
# Namespace creation is handled by ingest() internally — no stub needed here.
sys.modules["berdl_notebook_utils.setup_spark_session"].get_spark_session = None
sys.modules["berdl_notebook_utils.clients"].get_minio_client = None

import urllib3
from minio import Minio
from data_lakehouse_ingest import ingest
from get_spark_session import get_spark_session

logging.basicConfig(level=logging.INFO)
print("Imports OK.")

### Initialize Spark and MinIO

Checks SSH tunnels (ports 1337/1338), starts pproxy on :8123 if needed, connects Spark, and builds a MinIO client from `~/.mc/config.json`.

**Prerequisites:**
- SSH tunnels on ports 1337 and 1338 must be running (requires LBNL credentials — start manually).
- `mc` must be configured with a `berdl-minio` alias: `bash scripts/configure_mc.sh --berdl-proxy`
- `KBASE_AUTH_TOKEN` must be set in `.env` or the environment.

**Note on MinIO credentials:** This cell reads credentials from `~/.mc/config.json` (set up by `configure_mc.sh`). If `mc` is not yet installed, install it to `~/mc` and add to PATH:
```bash
curl -sSL https://dl.min.io/client/mc/release/linux-amd64/mc -o ~/mc && chmod +x ~/mc
export PATH="$HOME:$PATH"
```

In [ ]:
def _port_listening(port):
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

def _check_ssh_tunnels():
    missing = [p for p in [1337, 1338] if not _port_listening(p)]
    if missing:
        raise RuntimeError(
            f"[setup] SSH tunnel(s) missing on port(s): {missing}\n"
            "Start them in a terminal, then re-run this cell:\n"
            + "\n".join(
                f"  ssh -f -N -o ServerAliveInterval=60 -D {p} "
                "ac.<username>@login1.berkeley.kbase.us" for p in missing
            )
        )
    print("[setup] SSH tunnels OK (ports 1337, 1338)")

def _start_pproxy_if_needed():
    if _port_listening(8123):
        print("[setup] pproxy OK (port 8123)")
        return
    print("[setup] Starting pproxy...")
    subprocess.Popen(
        [sys.executable, "-c",
         "import sys, asyncio; "
         "sys.argv = ['pproxy', '-l', 'http://:8123', '-r', 'socks5://127.0.0.1:1338']; "
         "asyncio.set_event_loop(asyncio.new_event_loop()); "
         "from pproxy.server import main; main()"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(2)
    print("[setup] pproxy started" if _port_listening(8123) else "[setup] WARNING: pproxy may not have started")

_check_ssh_tunnels()
_start_pproxy_if_needed()

# MinIO client via mc alias
# Credentials are read from ~/.mc/config.json — run scripts/configure_mc.sh --berdl-proxy first
_mc_cfg      = json.loads((Path.home() / ".mc/config.json").read_text())
_alias       = _mc_cfg["aliases"]["berdl-minio"]
minio_client = Minio(
    endpoint    = _alias["url"].replace("https://", "").replace("http://", ""),
    access_key  = _alias["accessKey"],
    secret_key  = _alias["secretKey"],
    secure      = _alias["url"].startswith("https"),
    http_client = urllib3.ProxyManager("http://127.0.0.1:8123"),
)
print("[setup] MinIO client ready")

sys.modules["berdl_notebook_utils.clients"].get_minio_client = lambda **kw: minio_client

print("[spark] Connecting...")
spark = get_spark_session(berdl_proxy=True)
print("[spark] Spark session ready")

### Inspect source files

In [ ]:
import csv

print(f"Source directory: {DATA_DIR}\n")
for t in TABLES:
    f = DATA_DIR / t["source"]
    if not f.exists():
        raise FileNotFoundError(f"Missing source file: {f}")
    with open(f, newline="") as fh:
        header = next(csv.reader(fh, delimiter="\t"))
    print(f"  {t['name']:<15s}  {f.name:<35s}  {f.stat().st_size / 1024:.1f} KiB  columns: {header}")

### Upload source files to MinIO bronze

Skips any file whose size already matches in MinIO (safe to re-run after interruption).

In [ ]:
def _remote_size(key):
    try:
        return minio_client.stat_object(BUCKET, key).size
    except Exception as e:
        if any(tag in str(e) for tag in ("NoSuchKey", "does not exist", "404")):
            return -1
        raise

print(f"Uploading to: s3a://{BUCKET}/{BRONZE_DEST}/\n")
for t in TABLES:
    local       = DATA_DIR / t["source"]
    key         = f"{BRONZE_DEST}/{t['source']}"
    local_size  = local.stat().st_size
    remote_size = _remote_size(key)
    if remote_size == local_size:
        print(f"  {t['source']}: already in MinIO ({local_size:,} B) — skipping")
    else:
        print(f"  {t['source']}: uploading {local_size:,} B ...", end=" ", flush=True)
        minio_client.fput_object(BUCKET, key, str(local))
        print("done")

print("\nUpload complete.")

### Build config JSON

Writes the config to `~/.data_lakehouse/configs/` — this sandbox path is required by `ingest()`. Passing an arbitrary local path raises a sandbox violation error.

In [ ]:
config = {
    "dataset": DATASET,
    "paths": {
        "bronze_base": BRONZE_BASE,
    },
    "defaults": {
        "tsv": {"header": True, "delimiter": "\t", "inferSchema": False},
        # "csv": {"header": True, "delimiter": ",", "inferSchema": False},  # uncomment for CSV sources
    },
    "tables": [
        {
            "name":         t["name"],
            "enabled":      True,
            "schema_sql":   t["schema_sql"],
            "mode":         MODE,
            "partition_by": None,
            "bronze_path":  f"${{bronze_base}}/{t['source']}",
        }
        for t in TABLES
    ],
}

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print(f"Config written to: {CONFIG_PATH}\n")
print(json.dumps(config, indent=2))

### Pre-flight review

Review the plan printed below. When satisfied, set `CONFIRMED = True` in the Configuration cell and re-run from this cell onward.

In [ ]:
W = 70
print("=" * W)
print("PRE-FLIGHT PLAN")
print("=" * W)
print(f"  Namespace : {NAMESPACE}")
print(f"  Bronze    : {BRONZE_BASE}")
print(f"  Silver    : {SILVER_BASE}")
print(f"  Mode      : {MODE}")
print()
print(f"  {'Table':<15} {'Source file':<35} {'Schema columns'}")
print(f"  {'-'*15} {'-'*35} {'-'*20}")
for t in TABLES:
    cols = len(t["schema_sql"].split(",")) if t["schema_sql"] else "(inferred as STRING)"
    print(f"  {t['name']:<15} {t['source']:<35} {cols} columns")
print("=" * W)

if not CONFIRMED:
    raise RuntimeError(
        "\nSet CONFIRMED = True in the Configuration cell and re-run from here."
    )
print("CONFIRMED — proceeding with ingest.")

### Run ingest()

In [ ]:
report = ingest(str(CONFIG_PATH), spark=spark, minio_client=minio_client)
print(report)

### Verify row counts

In [ ]:
print("=" * 70)
print("VERIFICATION")
print("=" * 70)
spark.sql(f"SHOW TABLES IN `{NAMESPACE}`").show()

all_ok = True
print(f"{'Table':<15} {'Delta rows':>12} {'Expected':>12} {'Status'}")
print(f"{'-'*15} {'-'*12} {'-'*12} {'-'*8}")
for t in TABLES:
    f = DATA_DIR / t["source"]
    with open(f, newline="") as fh:
        expected = sum(1 for _ in fh) - 1      # subtract header row
    actual = spark.sql(f"SELECT COUNT(*) FROM `{NAMESPACE}`.`{t['name']}`").collect()[0][0]
    ok     = actual == expected
    all_ok = all_ok and ok
    print(f"{t['name']:<15} {actual:>12,} {expected:>12,} {'OK' if ok else 'MISMATCH'}")

print()
if all_ok:
    print("All row counts match. Ingest successful.")
    print(f"  Namespace : {NAMESPACE}")
    print(f"  Bronze    : {BRONZE_BASE}")
    print(f"  Silver    : {SILVER_BASE}")
else:
    print("Row count mismatch — check the ingest report above for rejected rows.")

In [ ]:
spark.stop()